<a href="https://colab.research.google.com/github/rudrarapubhavani/Bhavani/blob/main/Olive_Table_data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import pandas as pd
import os

# Download the dataset
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

# List files
print("Downloaded files:", os.listdir(path))

!cp -r {path}/* /content/
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Downloaded files: ['olist_customers_dataset.csv', 'olist_sellers_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_order_items_dataset.csv', 'olist_products_dataset.csv', 'olist_geolocation_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_payments_dataset.csv']


In [2]:
spark= SparkSession.builder.appName('Olive_set').getOrCreate()
path='/content/'


In [3]:
customer_df= spark.read.csv(path+'olist_customers_dataset.csv', header= True, inferSchema= True)
geolocation_df=spark.read.csv(path+'olist_geolocation_dataset.csv', header= True, inferSchema= True)
order_items_df= spark.read.csv(path+'olist_order_items_dataset.csv', header= True, inferSchema= True)
order_payments_df= spark.read.csv(path+'olist_order_payments_dataset.csv', header= True, inferSchema= True)
order_review_df= spark.read.csv(path+'olist_order_reviews_dataset.csv', header= True, inferSchema= True)
orders__df= spark.read.csv(path+'olist_orders_dataset.csv', header= True, inferSchema= True)
product_df= spark.read.csv(path+'olist_products_dataset.csv', header= True, inferSchema= True)
sellers_df= spark.read.csv(path+'olist_sellers_dataset.csv', header=True, inferSchema= True)
product_category= spark.read.csv(path+'product_category_name_translation.csv', header= True, inferSchema= True)


In [6]:
orders_df= spark.read.csv(path+'olist_orders_dataset.csv', header= True, inferSchema= True)
product_df= spark.read.csv(path+'olist_products_dataset.csv', header= True, inferSchema= True)
sellers_df= spark.read.csv(path+'olist_sellers_dataset.csv', header=True, inferSchema= True)
product_category= spark.read.csv(path+'product_category_name_translation.csv', header= True, inferSchema= True)

In [7]:
customer_df.createOrReplaceTempView('customer')
geolocation_df.createOrReplaceTempView('geolocation')
order_items_df.createOrReplaceTempView('order_item')
order_payments_df.createOrReplaceTempView('order_payment')
order_review_df.createOrReplaceTempView('order_review')
orders_df.createOrReplaceTempView('order')
product_df.createOrReplaceTempView('product')
sellers_df.createOrReplaceTempView('seller')
product_category.createOrReplaceTempView('product_category')


# Customer Table Analysis


In [8]:
spark.sql('''select * from customer limit(5)
''').show()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+



In [11]:
# customer distribution by state
spark.sql(''' select customer_state, count(customer_id) as total_customer from customer group by customer_state order by total_customer desc''').show()

+--------------+--------------+
|customer_state|total_customer|
+--------------+--------------+
|            SP|         41746|
|            RJ|         12852|
|            MG|         11635|
|            RS|          5466|
|            PR|          5045|
|            SC|          3637|
|            BA|          3380|
|            DF|          2140|
|            ES|          2033|
|            GO|          2020|
|            PE|          1652|
|            CE|          1336|
|            PA|           975|
|            MT|           907|
|            MA|           747|
|            MS|           715|
|            PB|           536|
|            PI|           495|
|            RN|           485|
|            AL|           413|
+--------------+--------------+
only showing top 20 rows


In [17]:
spark.sql('''select customer_state, count(distinct(customer_id)) as total_customers from customer
group by customer_state order by total_customers''').show(5)

+--------------+---------------+
|customer_state|total_customers|
+--------------+---------------+
|            RR|             46|
|            AP|             68|
|            AC|             81|
|            AM|            148|
|            RO|            253|
+--------------+---------------+
only showing top 5 rows


In [19]:
spark.sql(''' select * from order_item limit(5)''').show()

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

# Order_item table Analysis


In [21]:
# product highly purchased

spark.sql(''' select product_id , count(order_item_id) as total_orders from
order_item group by product_id order by total_orders desc ''').show(5)

+--------------------+------------+
|          product_id|total_orders|
+--------------------+------------+
|aca2eb7d00ea1a7b8...|         527|
|99a4788cb24856965...|         488|
|422879e10f4668299...|         484|
|389d119b48cf3043d...|         392|
|368c6c730842d7801...|         388|
+--------------------+------------+
only showing top 5 rows


In [24]:
# best seller
spark.sql('''
select seller_id, count(order_item_id) as total_orders from order_item
group by seller_id  order by total_orders desc limit(5)
''').show()



+--------------------+------------+
|           seller_id|total_orders|
+--------------------+------------+
|6560211a19b47992c...|        2033|
|4a3ca9315b744ce9f...|        1987|
|1f50f920176fa81da...|        1931|
|cc419e0650a3c5ba7...|        1775|
|da8622b14eb17ae28...|        1551|
+--------------------+------------+



In [25]:
# count of order items per id

spark.sql('''
select order_id, count(order_item_id) as total_items from order_item group by order_id
order by total_items desc limit(5)
''').show()

+--------------------+-----------+
|            order_id|total_items|
+--------------------+-----------+
|8272b63d03f5f79c5...|         21|
|1b15974a0141d54e3...|         20|
|ab14fdcfbe524636d...|         20|
|428a2f660dc84138d...|         15|
|9ef13efd6949e4573...|         15|
+--------------------+-----------+



In [26]:
# each seller have how many products

spark.sql('''
select seller_id, count(product_id) as total_products from order_item
group by seller_id order by total_products desc limit(5)
''').show()

+--------------------+--------------+
|           seller_id|total_products|
+--------------------+--------------+
|6560211a19b47992c...|          2033|
|4a3ca9315b744ce9f...|          1987|
|1f50f920176fa81da...|          1931|
|cc419e0650a3c5ba7...|          1775|
|da8622b14eb17ae28...|          1551|
+--------------------+--------------+



In [31]:
# highest sold product and respective seller id with total no of orders

spark.sql('''
select product_id, seller_id , count(order_item_id) as total_orders from order_item
group by product_id,seller_id order by total_orders desc limit(5)
''').show()

+--------------------+--------------------+------------+
|          product_id|           seller_id|total_orders|
+--------------------+--------------------+------------+
|aca2eb7d00ea1a7b8...|955fee9216a65b617...|         527|
|422879e10f4668299...|1f50f920176fa81da...|         484|
|99a4788cb24856965...|4a3ca9315b744ce9f...|         482|
|389d119b48cf3043d...|1f50f920176fa81da...|         392|
|368c6c730842d7801...|1f50f920176fa81da...|         388|
+--------------------+--------------------+------------+



In [37]:
# top 5 sellers by total revenue

spark.sql('''
select seller_id, sum(price) as revenue from order_item group by seller_id
order by revenue desc limit(5)
''').show()

+--------------------+------------------+
|           seller_id|           revenue|
+--------------------+------------------+
|4869f7a5dfa277a7d...|229472.62999999913|
|53243585a1d6dc264...| 222776.0499999998|
|4a3ca9315b744ce9f...| 200472.9199999981|
|fa1c13f2614d7b5c4...|194042.02999999968|
|7c67e1448b00f6e96...|187923.89000000118|
+--------------------+------------------+



In [42]:
# Products Expensive to ship to item price

spark.sql('''
select product_id, avg(price) as average_price, avg(freight_value) as shipping_cost from order_item
group by product_id having average_price < shipping_cost order by shipping_cost desc limit(5)
''').show()


+--------------------+-------------+-------------+
|          product_id|average_price|shipping_cost|
+--------------------+-------------+-------------+
|bc3c6d2a621414f2e...|        175.0|       299.16|
|c1e4bd7e1144f99ac...|        144.0|        269.0|
|d0d2f2b156caf4eec...|        139.0|       220.04|
|e00a60cd14eb7299b...|         74.9|       215.43|
|1b820caf50df248cf...|        169.9|        206.5|
+--------------------+-------------+-------------+



In [43]:
# multi item basket

spark.sql('''
select order_id, count(distinct(product_id)) as total_products from order_item
group by order_id order by total_products desc limit(5)
''').show()

+--------------------+--------------+
|            order_id|total_products|
+--------------------+--------------+
|ca3625898fbd48669...|             8|
|ad850e69fce9a512a...|             7|
|7d8f5bfd5aff64822...|             7|
|77df84f9195be22a4...|             7|
|2455cbeb73fd04b17...|             6|
+--------------------+--------------+



In [45]:
# Shipping Dead Lines. Per month how many items are shipped

order_items_df.printSchema()

spark.sql('''
select month(shipping_limit_date) as month, count(product_id) total_products from order_item
group by month order by total_products desc limit(5)
''').show()


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+-----+--------------+
|month|total_products|
+-----+--------------+
|    8|         13857|
|    5|         12915|
|    3|         11510|
|    7|         10788|
|    6|         10698|
+-----+--------------+



In [51]:
spark.sql('''
select date_format(shipping_limit_date , 'yyyy-MM') as shipping_month,
count(product_id) as total_products from order_item
group by shipping_month
order by shipping_month limit(5)
''').show()

+--------------+--------------+
|shipping_month|total_products|
+--------------+--------------+
|       2016-09|             4|
|       2016-10|           365|
|       2016-12|             1|
|       2017-01|           681|
|       2017-02|          1866|
+--------------+--------------+



# Order Payment VS Order Item Analysis


In [52]:
spark.sql('''
select * from order_payment limit(5)
''').show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+



In [55]:
# finding top payment with highest value


order_payments_df.printSchema()
order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



# Payment Re-concialation


Business Context: The accounting team needs to ensure that the total amount paid by the customer matches the actual cost of the items plus shipping. If there's a mismatch, it means there is a bug in the checkout system or a transaction error.


In [60]:
spark.sql(

          '''
          with Paid_amount as (select order_id , sum(payment_value) as amount_paid from order_payment group by order_id),
          Expected_amount as (select order_id, sum(price+ freight_value) as calculated_amount from order_item group by order_id)

          select i.order_id, i.amount_paid, p.calculated_amount,
          (i.amount_paid - p.calculated_amount) as descripancy from Paid_amount i
          join Expected_amount p on i.order_id= p.order_id
          where ROUND(i.amount_paid, 2) != ROUND(p.calculated_amount, 2)

          '''
).show()

+--------------------+-----------+------------------+--------------------+
|            order_id|amount_paid| calculated_amount|         descripancy|
+--------------------+-----------+------------------+--------------------+
|51a5a1d89c87180ca...|      92.91|             92.92|-0.01000000000000...|
|7a70b827ebc6ab85b...|     650.13|            650.14|-0.00999999999999...|
|5bc4d0aa53124192d...|     323.79|             323.8|-0.00999999999999...|
|8200c0f5298c25d0c...|     460.84|            460.85|-0.01000000000004...|
|911e92767c07ba992...|      49.73|             46.09|  3.6399999999999935|
|971d58b94f7bda9a7...|      83.89|             67.67|               16.22|
|04e00ba23c33890ea...|      53.41|              48.3|   5.109999999999999|
|0c87b31e37ed8557e...|     399.16|            399.15|0.010000000000047748|
|55881111d8b37a0e4...|     305.57|            305.58|-0.00999999999999...|
|73a15e8fc5de8485d...|     221.69|221.70000000000002|-0.01000000000001...|
|589afae8e1396e2a2...|   

In [61]:
spark.sql(
    '''
    select * from order_payment limit(5)
    '''
).show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+



# order Analysis


In [62]:
spark.sql(
    '''
    select * from order limit(5)
    '''
).show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [63]:
spark.sql('''
select customer_id, count(order_id) as total_orders from order group by customer_id
order by total_orders desc limit(5)
''').show()

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|f54a9f0e6b351c431...|           1|
|a9d37ddc8ba4d9f6d...|           1|
|2a1dfb647f32f4390...|           1|
|a78b75a2ad06180de...|           1|
|4f28355e5c17a4a42...|           1|
+--------------------+------------+



In [72]:
# time taken in approving the order

orders_df.printSchema()
spark.sql(
    '''
    select order_id, order_purchase_timestamp, order_approved_at,
    timestampdiff(hour, order_purchase_timestamp, order_approved_at) as delay_time_in_approving
     from order limit(5)
    '''
).show()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------+------------------------+-------------------+-----------------------+
|            order_id|order_purchase_timestamp|  order_approved_at|delay_time_in_approving|
+--------------------+------------------------+-------------------+-----------------------+
|e481f51cbdc54678b...|     2017-10-02 10:56:33|2017-10-02 11:07:15|                      0|
|53cdb2fc8bc7dce0b...|     2018-07-24 20:41:37|2018-07-26 03:24:27|                     30|
|47770eb9100c2d0c4...|     2018-08-08 08:38:49|2018-08-08 08:55:23|                      0|
|949d5b44dbf5

In [85]:
# count count of orders deleved late VS count of orders delivered early

spark.sql(
    '''
    select  order_status, count(order_id) from order group by order_status
    limit(5)

    '''
).show()

spark.sql(
    '''
    select order_status, order_id, order_estimated_delivery_date,order_delivered_customer_date,
    timestampdiff(hour, order_estimated_delivery_date,order_delivered_customer_date) as delay
    from order  limit(5)
    '''
).show()


spark.sql('''
SELECT
    CASE
        WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 'Late'
        ELSE 'On Time / Early'
    END AS delivery_performance,
    COUNT(order_id) AS total_orders
FROM order
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
GROUP BY
    CASE
        WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 'Late'
        ELSE 'On Time / Early'
    END
''').show()

+------------+---------------+
|order_status|count(order_id)|
+------------+---------------+
|     shipped|           1107|
|    canceled|            625|
|    approved|              2|
|    invoiced|            314|
|     created|              5|
+------------+---------------+

+------------+--------------------+-----------------------------+-----------------------------+-----+
|order_status|            order_id|order_estimated_delivery_date|order_delivered_customer_date|delay|
+------------+--------------------+-----------------------------+-----------------------------+-----+
|   delivered|e481f51cbdc54678b...|          2017-10-18 00:00:00|          2017-10-10 21:25:13| -170|
|   delivered|53cdb2fc8bc7dce0b...|          2018-08-13 00:00:00|          2018-08-07 15:27:45| -128|
|   delivered|47770eb9100c2d0c4...|          2018-09-04 00:00:00|          2018-08-17 18:06:29| -413|
|   delivered|949d5b44dbf5de918...|          2017-12-15 00:00:00|          2017-12-02 00:28:42| -311|
|   de

In [91]:

spark.sql(
    '''
    select case
    when order_delivered_customer_date > order_estimated_delivery_date then 'late'
    else 'Early/ On-Time' end as delivery_status ,
    count(order_id) as total_orders from order
    where order_status == 'delivered' and order_delivered_customer_date is not null
    group by case
    when order_delivered_customer_date > order_estimated_delivery_date then 'late'
    else 'Early/ On-Time' end


    '''
).show(5)

+---------------+------------+
|delivery_status|total_orders|
+---------------+------------+
|           late|        7826|
| Early/ On-Time|       88644|
+---------------+------------+



In [102]:
# average time taken to pack the order

spark.sql(
    '''
    select order_status,
    round(avg(datediff(order_delivered_carrier_date, order_approved_at)),2) as packing_time,
    max(datediff(order_delivered_carrier_date, order_approved_at)) as max_packing_time
    from order where order_delivered_carrier_date is not null
    group by order_status
    limit(5)
    '''
).show()

+------------+------------+----------------+
|order_status|packing_time|max_packing_time|
+------------+------------+----------------+
|     shipped|        3.23|              40|
|    canceled|        3.11|              38|
|   delivered|         2.7|             126|
+------------+------------+----------------+



In [108]:
# Which day have highest orders

spark.sql(
    '''
    select count(order_id) total_orders, date_format(order_purchase_timestamp, 'EEEE') as day
    from order
    where order_purchase_timestamp is not null
    group by date_format(order_purchase_timestamp, 'EEEE')
    order by total_orders desc


    '''
).show()

+------------+---------+
|total_orders|      day|
+------------+---------+
|       16196|   Monday|
|       15963|  Tuesday|
|       15552|Wednesday|
|       14761| Thursday|
|       14122|   Friday|
|       11960|   Sunday|
|       10887| Saturday|
+------------+---------+



# Product Analysis

In [109]:
spark.sql(
    '''
    select * from product limit(5)
    '''
).show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [110]:
product_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [115]:
# Average no of photos and product description

spark.sql(
    '''
    select product_category_name,
    avg(product_photos_qty) as avg_photos,
    avg(product_description_lenght) as avg_descrition_length
    from product group by product_category_name
    order by avg_descrition_length desc
    limit(5)
    '''
).show()

+---------------------+------------------+---------------------+
|product_category_name|        avg_photos|avg_descrition_length|
+---------------------+------------------+---------------------+
|                  pcs|               2.7|   2128.8333333333335|
|    moveis_escritorio|1.3300970873786409|   1352.7669902912621|
|      livros_tecnicos|1.0975609756097562|   1351.5284552845528|
|         beleza_saude|1.6243862520458265|   1136.9337152209494|
|            alimentos|1.9390243902439024|   1136.5121951219512|
+---------------------+------------------+---------------------+

